In [1]:
import torch
import pandas as pd 
import numpy as np 
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
train_dataset=pd.read_csv("/kaggle/input/datasets/atulanandjha/imdb-50k-movie-reviews-test-your-bert/train.csv")
test_dataset=pd.read_csv("/kaggle/input/datasets/atulanandjha/imdb-50k-movie-reviews-test-your-bert/test.csv")

train_dataset["text"] = train_dataset["text"].str.lower()
train_dataset["text"] = train_dataset["text"].str.replace(r"<br\s*/?>", " ", regex=True)
test_dataset["text"] = test_dataset["text"].str.lower()
test_dataset["text"] = test_dataset["text"].str.replace(r"<br\s*/?>", " ", regex=True)
train_dataset["sentiment"] = train_dataset["sentiment"].map({"neg": 0, "pos": 1})
test_dataset["sentiment"] = test_dataset["sentiment"].map({"neg": 0, "pos": 1})
from collections import Counter

# 1. Build vocab from the ENTIRE training corpus
all_tokens = []
for text in train_dataset["text"]:
    all_tokens.extend(text.split())

word_counts = Counter(all_tokens)
# keep only words appearing at least twice (cuts noise/typos, shrinks vocab)
vocab_words = [w for w, c in word_counts.items() if c >= 2]

# 2. Special tokens at fixed indices
num_encoding = {"<PAD>": 0, "<UNK>": 1}
for i, w in enumerate(vocab_words, start=2):
    num_encoding[w] = i

vocab_size = len(num_encoding)
print(vocab_size)  # sanity check — expect tens of thousands
# 3. Encode + pad/truncate to max_length
def encode_review(text, num_encoding, max_length=512):
    tokens = text.split()
    ids = [num_encoding.get(t, num_encoding["<UNK>"]) for t in tokens]
    if len(ids) < max_length:
        ids = ids + [num_encoding["<PAD>"]] * (max_length - len(ids))
    else:
        ids = ids[:max_length]
    return ids
# 4. Dataset + DataLoaderclass IMDBDataset(Dataset):
class IMDBDataset(Dataset):
    def __init__(self, dataframe, num_encoding, max_length=512):
        self.texts = dataframe["text"].values
        self.labels = dataframe["sentiment"].values
        self.num_encoding = num_encoding
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode_review(self.texts[idx], self.num_encoding, self.max_length)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

train_ds = IMDBDataset(train_dataset, num_encoding, max_length=512)
test_ds = IMDBDataset(test_dataset, num_encoding, max_length=512)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)


class SentimentLSTM(nn.Module):
    def __init__(self,vocab_size,embedding_dim=128,hidden_dim=128,pad_idx=0):
      super().__init__()
      self.vocab_size=vocab_size
      self.embedding_dim=embedding_dim
      self.hidden_dim=hidden_dim
      self.pad_idx=pad_idx
      self.embed=nn.Embedding(vocab_size,embedding_dim,padding_idx=pad_idx)
      self.lstm=nn.LSTM(embedding_dim,hidden_dim,batch_first=True)
      self.linear=nn.Linear(hidden_dim,1)
      
    def forward(self,x):
        x=self.embed(x)
        output,(hidden,cell)=self.lstm(x)
        hidden=hidden.squeeze(0)
        logits =self.linear(hidden)
        logits=logits.squeeze(0)
        return logits
        


    
model = SentimentLSTM(vocab_size=vocab_size, embedding_dim=128, hidden_dim=128, pad_idx=0)


95999


train_dataset["sentiment"] = train_dataset["sentiment"].map({
    "neg": 0,
    "pos": 1
})

test_dataset["sentiment"] = test_dataset["sentiment"].map({
    "neg": 0,
    "pos": 1
})